In [3]:
import pandas as pd

In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data"

tickets = pd.read_csv(DATA_PATH / "rideops_support_tickets.csv")

data_dictionary = pd.read_csv( DATA_PATH / "rideops_data_dictionary.csv")

In [5]:
print("Support Tickets:", tickets.shape)
print("Data Dictionary:", data_dictionary.shape)

Support Tickets: (1444, 20)
Data Dictionary: (19, 2)


In [6]:
status_counts = tickets['status'].value_counts()

status_distribution = pd.DataFrame({
    'Ticket Count': status_counts,
    'Percentage': (status_counts / len(tickets) * 100).round(2)
})

display(status_distribution)

,Ticket Count,Percentage
status,,
Closed,1396,96.68
Open - with L1,29,2.01
Open - with Engineering,19,1.32


In [10]:
date_columns = [
    'created_at',
    'first_response_at',
    'first_closed_at',
    'final_closed_at'
]

for col in date_columns:
    tickets[col] = pd.to_datetime(tickets[col], errors='coerce')

In [11]:
tickets['response_time_hrs'] = (
    tickets['first_response_at'] - tickets['created_at']
).dt.total_seconds() / 3600

tickets['first_closure_time_hrs'] = (
    tickets['first_closed_at'] - tickets['created_at']
).dt.total_seconds() / 3600

In [12]:
print("Average response time:",
      tickets['response_time_hrs'].mean())

print("Average first closure time:",
      tickets['first_closure_time_hrs'].mean())

Average response time: 4.686055130877924
Average first closure time: 41.32504755111745


In [13]:
average_csat = tickets['csat_score'].mean()

print("Average CSAT:", round(average_csat, 2), "/ 5")
print("CSAT responses:", tickets['csat_score'].count())

Average CSAT: 3.7 / 5
CSAT responses: 648


In [14]:
escalation_counts = tickets['escalated_to_eng'].value_counts()

escalation_distribution = pd.DataFrame({
    'Ticket Count': escalation_counts,
    'Percentage': (
        escalation_counts / len(tickets) * 100
    ).round(2)
})

display(escalation_distribution)

,Ticket Count,Percentage
escalated_to_eng,,
No,958,66.34
Yes,486,33.66


In [15]:
priority_distribution = (
    tickets['priority']
    .value_counts()
    .to_frame('Ticket Count')
)

priority_distribution['Percentage'] = (
    priority_distribution['Ticket Count'] / len(tickets) * 100
).round(2)

display(priority_distribution)

,Ticket Count,Percentage
priority,,
P3,742,51.39
P4,371,25.69
P2,264,18.28
P1,67,4.64


In [16]:
channel_distribution = (
    tickets['channel']
    .value_counts()
    .to_frame('Ticket Count')
)

channel_distribution['Percentage'] = (
    channel_distribution['Ticket Count'] / len(tickets) * 100
).round(2)

display(channel_distribution)

,Ticket Count,Percentage
channel,,
Email,553,38.30
Client portal,377,26.11
Phone,292,20.22
WhatsApp,222,15.37


In [19]:
total_tickets = len(tickets)

# Status
closed_tickets = (tickets['status'] == 'Closed').sum()
open_tickets = total_tickets - closed_tickets

# Escalations
escalated_tickets = (tickets['escalated_to_eng'] == 'Yes').sum()

# Summary table
summary = pd.DataFrame({
    'Metric': [
        'Total Tickets',
        'Open Tickets',
        'Open Tickets (%)',
        'Closed Tickets',
        'Closed Tickets (%)',
        'Average First Response Time (hrs)',
        'Average First Closure Time (hrs)',
        'Average CSAT Score',
        'CSAT Responses',
        'Escalated Tickets',
        'Escalated Tickets (%)'
    ],
    'Value': [
        total_tickets,
        open_tickets,
        round(open_tickets / total_tickets * 100, 2),
        closed_tickets,
        round(closed_tickets / total_tickets * 100, 2),
        round(tickets['response_time_hrs'].mean(), 2),
        round(tickets['first_closure_time_hrs'].mean(), 2),
        round(tickets['csat_score'].mean(), 2),
        tickets['csat_score'].count(),
        escalated_tickets,
        round(escalated_tickets / total_tickets * 100, 2)
    ]
})

display(summary)

,Metric,Value
0,Total Tickets,1444.00
1,Open Tickets,48.00
2,Open Tickets (%),3.32
3,Closed Tickets,1396.00
4,Closed Tickets (%),96.68
5,Average First Response Time (hrs),4.69
6,Average First Closure Time (hrs),41.33
7,Average CSAT Score,3.70
8,CSAT Responses,648.00
9,Escalated Tickets,486.00


In [ ]:
engineering_outcomes = tickets[
    tickets['escalated_to_eng'] == 'Yes'
]['eng_outcome'].value_counts()

engineering_summary = engineering_outcomes.to_frame(
    name='Ticket Count'
)

engineering_summary['Percentage'] = (
    engineering_summary['Ticket Count'] /
    escalated_tickets * 100
).round(2)

display(engineering_summary)

Tickets escalated: 486
Tickets fixed by engineering: 211
Tickets returned to L1: 275


In [22]:
returned_pct_total = (
    returned_to_l1 / len(tickets) * 100
)
returned_pct_escalated = (
    returned_to_l1 / escalated_tickets * 100
)
print(f"Returned to L1 (% of total tickets): {returned_pct_total:.2f}%")

print(f"Returned to L1 (% of escalated tickets): {returned_pct_escalated:.2f}%")

Returned to L1 (% of total tickets): 19.04%
Returned to L1 (% of escalated tickets): 56.58%
